In [ ]:
# CELLA IMPORTAZIONI LIBRERIE PER MAC

import os
import glob
import logging
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from keras.models import load_model

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0' 
logging.getLogger('tensorflow').setLevel(logging.ERROR)

import tensorflow as tf
 
tf.get_logger().setLevel('ERROR')
tf.autograph.set_verbosity(0)

# Import di Keras 
from keras import layers, models, losses
from keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

In [ ]:
# DATA ENGINE V9 

def load_and_process_all_files(file_list, alpha=0.20):
    X_all, Y_all = [], []
    print(f"Inizio caricamento ed EMA Decluttering di {len(file_list)} file...")
    
    for i, file_path in enumerate(file_list):
        data = np.load(file_path)
        raw_iq = data['radar_cir_iq']   
        people_xy = data['people_xy']   
        people_mask = data['people_mask'] 
        T = raw_iq.shape[0]             
        
        mag = np.sqrt(raw_iq[..., 0]**2 + raw_iq[..., 1]**2)
        mag_reshaped = mag.reshape(T, 1, 120, 18) 
        
        # Inizializziamo il background per l'EMA
        bg = np.copy(mag_reshaped[0])
        decluttered = np.zeros_like(mag_reshaped)
        
        # L'EMA 
        for t in range(T):
            bg = alpha * mag_reshaped[t] + (1 - alpha) * bg
            decluttered[t] = np.abs(mag_reshaped[t] - bg)
        
        # Flatten delle coordinate e concatenazione con la mask (12 valori totali)
        flat_coords = people_xy.reshape(T, 8)
        combined_target = np.concatenate([flat_coords, people_mask], axis=1)

        X_all.append(decluttered)
        Y_all.append(combined_target)
        
        print(f"File {i+1}/{len(file_list)} processato. ({T} frame pre-calcolati)")

    X = np.concatenate(X_all, axis=0).astype(np.float32)
    Y = np.concatenate(Y_all, axis=0).astype(np.float32)
    return X, Y

# ==============================================================================
# SPLIT DIVERSI SEGUENDO DIVERSI CRITERI
# ==============================================================================
# Split1 (78% train e 22% val), > windows con 3/4 persone in train
val_indices = [23, 20, 3, 15, 7, 11] 
train_indices = [22, 16, 17, 18, 19, 21, 0, 1, 2, 4, 12, 13, 14, 5, 6, 8, 9, 10]
# Split2 (80% train e 22% val), stesso cocetto di split 1 "STRESS TEST sul MULTIPATH"
val_indices = [23, 20, 0, 13, 9] 
train_indices = [22, 16, 17, 18, 19, 21, 1, 2, 3, 4, 12, 14, 15, 5, 6, 7, 8, 10, 11]
# Split3 (78% train e 22% val) > equilibrato tra train e val 
#val_indices = [22, 20, 2, 12, 14, 6] 
#train_indices = [23, 16, 18, 19, 21, 0, 1, 3, 4, 13, 15, 5, 7, 8, 9, 10, 11, 17]

#tutti_i_file = glob.glob("dataset/data/*.npz")
tutti_i_file = glob.glob("dataset/*.npz")
train_files = [f for f in tutti_i_file if int(os.path.basename(f).replace("window_", "").replace(".npz", "")) in train_indices]
val_files = [f for f in tutti_i_file if int(os.path.basename(f).replace("window_", "").replace(".npz", "")) in val_indices]

# ==============================================================================
# ESECUZIONE DEL MOTORE 
# ==============================================================================
print("\n--- PREPARAZIONE TRAINING SET ---")
X_train, Y_train = load_and_process_all_files(train_files)

print("\n--- PREPARAZIONE VALIDATION SET ---")
X_val, Y_val = load_and_process_all_files(val_files)

print("\n==================================================")
print(f"DATI TOTALI PRONTI IN RAM!")
print(f"Totale FRAME individuali di Train:      {X_train.shape[0]}")
print(f"Totale FRAME individuali di Validation: {X_val.shape[0]}")
print("==================================================")

In [ ]:
# HUNGARIAN Loss PER MAC

import itertools
PERM_INDICES = tf.constant(list(itertools.permutations([0, 1, 2, 3])), dtype=tf.int32)

def hungarian_total_loss(y_true, y_pred):
    y_true_coords = tf.reshape(y_true[:, :8], (-1, 4, 2))
    y_pred_coords = tf.reshape(y_pred[:, :8], (-1, 4, 2))
    y_true_mask = tf.reshape(y_true[:, 8:], (-1, 4, 1))
    y_pred_mask = tf.reshape(y_pred[:, 8:], (-1, 4, 1))

    y_pred_coords_perm = tf.gather(y_pred_coords, PERM_INDICES, axis=1) 
    y_pred_mask_perm = tf.gather(y_pred_mask, PERM_INDICES, axis=1)

    y_true_coords_exp = tf.expand_dims(y_true_coords, 1) 
    y_true_mask_exp = tf.expand_dims(y_true_mask, 1)

    sq_diff = tf.square(y_true_coords_exp - y_pred_coords_perm)
    coords_cost = tf.reduce_sum(sq_diff * y_true_mask_exp, axis=[2, 3]) 
    
    num_valid_people = tf.reduce_sum(y_true_mask_exp[:, 0, :, :], axis=[1, 2]) + 1e-6
    coords_cost_norm = coords_cost / tf.expand_dims(num_valid_people, axis=-1)

    #bce = tf.keras.backend.binary_crossentropy(y_true_mask_exp, y_pred_mask_perm)
    y_pred_mask_perm_safe = tf.clip_by_value(y_pred_mask_perm, 1e-7, 1.0 - 1e-7)
    bce = tf.keras.backend.binary_crossentropy(y_true_mask_exp, y_pred_mask_perm_safe)
    mask_cost_norm = tf.reduce_mean(bce, axis=[2, 3]) 

    total_cost = coords_cost_norm + (1.5 * mask_cost_norm) 
    return tf.reduce_min(total_cost, axis=1) 

def hungarian_rmse_metres(y_true, y_pred):
    y_true_coords = tf.reshape(y_true[:, :8], (-1, 4, 2))
    y_pred_coords = tf.reshape(y_pred[:, :8], (-1, 4, 2))
    y_true_mask = tf.reshape(y_true[:, 8:], (-1, 4, 1))

    y_pred_coords_perm = tf.gather(y_pred_coords, PERM_INDICES, axis=1)
    y_true_coords_exp = tf.expand_dims(y_true_coords, 1)
    y_true_mask_exp = tf.expand_dims(y_true_mask, 1)

    sq_diff = tf.square(y_true_coords_exp - y_pred_coords_perm)
    coords_cost = tf.reduce_sum(sq_diff * y_true_mask_exp, axis=[2, 3])
    
    min_coords_cost = tf.reduce_min(coords_cost, axis=1)
    
    num_valid_people = tf.reduce_sum(y_true_mask_exp[:, 0, :, :], axis=[1, 2]) + 1e-6
    
    return tf.sqrt(min_coords_cost / num_valid_people)

def hungarian_mask_acc(y_true, y_pred):
    y_true_mask = tf.reshape(y_true[:, 8:], (-1, 4, 1))
    y_pred_mask = tf.reshape(y_pred[:, 8:], (-1, 4, 1))
    y_pred_mask_perm = tf.gather(y_pred_mask, PERM_INDICES, axis=1)
    
    y_true_coords = tf.reshape(y_true[:, :8], (-1, 4, 2))
    y_pred_coords = tf.reshape(y_pred[:, :8], (-1, 4, 2))
    y_pred_coords_perm = tf.gather(y_pred_coords, PERM_INDICES, axis=1)
    y_true_coords_exp = tf.expand_dims(y_true_coords, 1)
    y_true_mask_exp = tf.expand_dims(y_true_mask, 1)

    sq_diff = tf.square(y_true_coords_exp - y_pred_coords_perm)
    coords_cost = tf.reduce_sum(sq_diff * y_true_mask_exp, axis=[2, 3])
    num_valid_people = tf.reduce_sum(y_true_mask_exp[:, 0, :, :], axis=[1, 2]) + 1e-6
    coords_cost_norm = coords_cost / tf.expand_dims(num_valid_people, axis=-1)
    
    #bce = tf.keras.backend.binary_crossentropy(y_true_mask_exp, y_pred_mask_perm)
    y_pred_mask_perm_safe = tf.clip_by_value(y_pred_mask_perm, 1e-7, 1.0 - 1e-7)
    bce = tf.keras.backend.binary_crossentropy(y_true_mask_exp, y_pred_mask_perm_safe)
    mask_cost_norm = tf.reduce_mean(bce, axis=[2, 3])
    
    total_cost = coords_cost_norm + (1.5 * mask_cost_norm)
    best_perm_idx = tf.argmin(total_cost, axis=1, output_type=tf.int32)
    
    batch_size = tf.shape(y_pred)[0]
    gather_nd_indices = tf.stack([tf.range(batch_size, dtype=tf.int32), best_perm_idx], axis=1)
    best_mask_pred = tf.gather_nd(y_pred_mask_perm, gather_nd_indices)
    
    return tf.reduce_mean(tf.keras.metrics.binary_accuracy(y_true_mask, best_mask_pred))

In [ ]:
# ARCHITETTURA EEAI-NET V2 

def squeeze_excite_block_2d(x, filters, r=8):
    """Meccanismo di Attenzione spaziale basato su Squeeze-and-Excitation"""
    # Squeeze
    se = layers.GlobalAveragePooling2D()(x)
    # Excitation
    se = layers.Dense(max(1, filters // r), activation='relu', use_bias=False)(se)
    se = layers.Dense(filters, activation='sigmoid', use_bias=False)(se)
    # Reshape 
    se = layers.Reshape((1, 1, filters))(se)
    return layers.Multiply()([x, se])

def residual_reduction_module_2d(x, filters, r=8, name_prefix=""):
    """
    RRM: Red(Res(x)) 
    Unisce una skip connection pesata dal SE block e un dimezzamento 
    parallelo della dimensione temporale (range bins).
    """
    # Residual Branch 
    res = layers.Conv2D(filters, kernel_size=(1, 3), padding='same', activation='relu', 
                        name=f"{name_prefix}_res_conv")(x)
    res = squeeze_excite_block_2d(res, filters, r=r)
    res = layers.Add(name=f"{name_prefix}_res_add")([res, x]) # Skip connection

    # Reduction Branch 
    red1 = layers.Conv2D(filters, kernel_size=(1, 3), strides=(1, 2), padding='same', 
                         activation='relu', name=f"{name_prefix}_red_conv1")(res)
    red2 = layers.Conv2D(filters, kernel_size=(1, 1), strides=(1, 2), padding='same', 
                         activation='relu', name=f"{name_prefix}_red_conv2")(res)
    
    out = layers.Add(name=f"{name_prefix}_red_add")([red1, red2])
    return out

def build_eeai_model_v2_rrm(n_radars=6, n_antennas=3, n_bins=120):
    input_channels = n_radars * n_antennas
    inputs = layers.Input(shape=(1, n_bins, input_channels), name="radar_input")
    
    F = 48 
    r = 8  
    
    # Feature Extraction 
    x = layers.Conv2D(F, kernel_size=(1, 5), padding='same', activation='relu', name="init_conv")(inputs)
    
    # Cascata di Residual Reduction Modules
    x = residual_reduction_module_2d(x, filters=F, r=r, name_prefix="rrm1") # Bins: 120 -> 60
    x = residual_reduction_module_2d(x, filters=F, r=r, name_prefix="rrm2") # Bins: 60 -> 30
    x = residual_reduction_module_2d(x, filters=F, r=r, name_prefix="rrm3") # Bins: 30 -> 15
    x = residual_reduction_module_2d(x, filters=F, r=r, name_prefix="rrm4") # Bins: 15 -> 8
    
    x = layers.Flatten(name="flatten_features")(x)
    x = layers.Dropout(0.25, name="dropout_features")(x)
    
    common_feat = layers.Dense(128, activation='relu', name="dense_shared")(x)
    
    # Multi-Head Output 
    coords_output = layers.Dense(8, activation='linear', name="coords_head")(common_feat)
    mask_output = layers.Dense(4, activation='sigmoid', name="mask_head")(common_feat)
    
    combined_output = layers.Concatenate(axis=1, name="combined_output")([coords_output, mask_output])
    
    return models.Model(inputs=inputs, outputs=combined_output, name="EEAI_Net_V2_RRM")

In [ ]:
# ADDESTRAMENTO MODELLO V2 (RRM)

# Inizializzazione del nuovo modello
model_rrm = build_eeai_model_v2_rrm()

# Compilazione 
model_rrm.compile(
    optimizer='adam',
    loss=hungarian_total_loss, 
    metrics=[hungarian_rmse_metres, hungarian_mask_acc]
)

checkpoint_rrm = ModelCheckpoint("eeai_best_model_romano_rrm1.keras", monitor="val_loss", save_best_only=True, verbose=1)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1)
early_stop = EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True, verbose=1)

EPOCHS = 50 

print("\n--- INIZIO ADDESTRAMENTO MODELLO RRM ---")
history_rrm = model_rrm.fit(
    X_train, Y_train,
    validation_data=(X_val, Y_val),
    batch_size=32,
    shuffle=True,
    epochs=EPOCHS,
    callbacks=[checkpoint_rrm, reduce_lr, early_stop], 
    verbose=1
)
print("--- ADDESTRAMENTO COMPLETATO ---")

In [ ]:
# MODEL SUMMARY PER EMBEDDED

def embedded_summary(model, input_shape=(1, 120, 18)):
    
    # Calcola i parametri statici (Flash)
    total_params = model.count_params()
    estimated_flash_kb = (total_params * 4) / 1024
    
    # Calcola il picco di memoria dinamica (SRAM/Tensor Arena)
    max_layer_ram_kb = 0
    for layer in model.layers:
        if layer.__class__.__name__ == 'InputLayer' or not hasattr(layer, 'output_shape'):
            continue
            
        output_shape = layer.output_shape
        if isinstance(output_shape, list):
            num_elements = sum([np.prod([dim for dim in shape[1:] if dim is not None]) for shape in output_shape])
        else:
            num_elements = np.prod([dim for dim in output_shape[1:] if dim is not None])
            
        layer_ram_kb = (num_elements * 4) / 1024
        if layer_ram_kb > max_layer_ram_kb:
            max_layer_ram_kb = layer_ram_kb

    input_elements = np.prod(input_shape)
    input_ram_kb = (input_elements * 4) / 1024
    peak_arena_kb = input_ram_kb + max_layer_ram_kb

    # Stampa il verdetto 
    print("============================================")
    print("   REPORT REQUISITI HARDWARE (STIMA FLOAT32)   ")
    print("============================================")
    print(f" Memoria FLASH stimata : {estimated_flash_kb:.2f} KB  (Limite : < 800 KB)")
    print(f" Memoria SRAM stimata  : ~{peak_arena_kb:.2f} KB (Limite : < 400 KB)")
    print(" Operazioni Ricorrenti : ASSENTI (RNN/LSTM/GRU non rilevate)")
    print(" Nota sulla Quantizz.  : Raccomandata INT8 per ESP32-S3 (ridurrà la RAM di ~4x)")
    print("============================================\n")

embedded_summary(model_rrm)
# model_rrm.summary() 

In [ ]:
# VISUALIZZATORE 3.1 

file_target = "dataset/window_000000.npz"
if not os.path.exists(file_target):
    print(f"ERRORE: Non trovo il file {file_target}")
else:
    data = np.load(file_target)
    raw_iq = data['radar_cir_iq'] 
    gt_coords = data['people_xy'] 
    gt_mask = data['people_mask'] 
    T = raw_iq.shape[0]

    print("Elaborazione filtri e previsioni in corso (V8)...")
    mag = np.sqrt(raw_iq[..., 0]**2 + raw_iq[..., 1]**2).reshape(T, 1, 120, 18)
    decluttered = np.zeros_like(mag)
    bg = np.copy(mag[0])
    alpha = 0.20
    for t in range(T):
        bg = alpha * mag[t] + (1 - alpha) * bg
        decluttered[t] = np.abs(mag[t] - bg)

    print("Caricamento dei pesi migliori del modello RRM dal file .keras ...")
    
    model_rrm_loaded = load_model(
        "eeai_best_model_romano_rrm1.keras",
        custom_objects={
            "hungarian_total_loss": hungarian_total_loss,
            "hungarian_rmse_metres": hungarian_rmse_metres, 
            "hungarian_mask_acc": hungarian_mask_acc
        }
    )

    preds = model_rrm_loaded.predict(decluttered, verbose=0)
    
    p_coords = preds[:, :8].reshape(T, 4, 2)
    p_mask = preds[:, 8:]
    
    print("Dati pronti! Inizializzazione Radar...")

    out = widgets.Output() 

    def draw_frame(frame_idx, soglia):
        with out:
            clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(4.8, 7.2))
            ax.set_xlim(-0.5, 5.3); ax.set_ylim(-0.5, 7.7)
            ax.grid(True, linestyle=':', alpha=0.6)
            
            ax.set_title(f"Radar V9 (RRM) | Frame: {frame_idx}/{T-1} | Window: 5", fontsize=14, fontweight='bold')

            stanza = plt.Rectangle((0, 0), 4.8, 7.2, linewidth=3, edgecolor='navy', facecolor='whitesmoke')
            ax.add_patch(stanza)

            for i in range(4):
                is_present = bool(gt_mask[frame_idx, i] > 0.5)
                if is_present:
                    rx, ry = gt_coords[frame_idx, i]
                    ax.scatter(rx, ry, c='limegreen', s=120, edgecolors='black', marker='o', label='REALE (GT)' if i==0 else "")
                    ax.text(rx, ry + 0.2, f"P{i+1}", color='darkgreen', fontweight='bold', ha='center')

                conf = float(p_mask[frame_idx, i])
                if conf >= soglia:
                    px, py = p_coords[frame_idx, i]
                    alpha_val = max(0.3, conf)
                    ax.scatter(px, py, c='red', s=100, marker='X', edgecolors='darkred', alpha=alpha_val, label='PREDETTO' if i==0 else "")
                    ax.text(px, py - 0.3, f"{conf*100:.0f}%", color='red', fontsize=10, ha='center', fontweight='bold')

            handles, labels = ax.get_legend_handles_labels()
            by_label = dict(zip(labels, handles))
            if by_label:
                ax.legend(by_label.values(), by_label.keys(), loc='upper right', frameon=True, shadow=True)

            plt.xlabel("X (Metri)"); plt.ylabel("Y (Metri)")
            plt.tight_layout(); plt.show()

    slider_frame = widgets.IntSlider(value=500, min=10, max=T-1, step=1, description='Frame:')
    slider_soglia = widgets.FloatSlider(value=0.50, min=0.1, max=0.99, step=0.05, description='Soglia:')

    def on_change(change):
        draw_frame(slider_frame.value, slider_soglia.value)

    slider_frame.observe(on_change, names='value')
    slider_soglia.observe(on_change, names='value')

    controls = widgets.VBox([slider_frame, slider_soglia])
    controls.layout.margin = '20px 20px 20px 0px' 
    ui = widgets.HBox([controls, out])
    
    display(ui)
    draw_frame(slider_frame.value, slider_soglia.value)